# RAG Bayesian Model Demo: The RAT Framework

This notebook demonstrates the **RAT (Retrieval-Augmented Text evaluation)** framework by:
1. Importing the `RAG_BayesianModel` from `implementation.py`.
2. Simulating two RAG systems with different behavioral patterns (Conservative vs. Aggressive) that have **identical end-to-end success rates** but **different internal behaviors**.
3. Updating the Bayesian model with simulated data to show how conditional decomposition reveals these differences.
4. Visualizing the posterior distributions of retrieval success, generator actions, and answer correctness.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from implementation import (
    RAG_BayesianModel,
    R_RETRIEVAL_SUCCESS,
    R_RETRIEVAL_FAILURE,
    A_ANSWER,
    A_ABSTAIN,
    A_GUESS,
    C_CORRECT,
    C_INCORRECT
)

# 1. Simulate Data for Two RAG Systems

We create two synthetic datasets:
- **System A (Conservative)**: High retrieval success, tends to abstain when retrieval fails.
- **System B (Aggressive)**: Low retrieval success, tends to guess/answer even when retrieval fails.

Both systems are tuned to have roughly the same **marginal task success rate** (final answer correctness), but their internal behavior differs significantly.

In [ ]:
def generate_simulated_data(n_samples, retrieval_success_prob, action_probs_by_retrieval, correctness_probs_by_ra):
    """
    Generate synthetic RAG pipeline data.
    
    Args:
        n_samples: Number of samples.
        retrieval_success_prob: P(Retrieval Success)
        action_probs_by_retrieval: Dict {(r, a): prob}
        correctness_probs_by_ra: Dict {(r, a, c): prob}
    
    Returns:
        List of tuples (r, a, c)
    """
    data = []
    for _ in range(n_samples):
        # Sample retrieval success
        r = R_RETRIEVAL_SUCCESS if np.random.random() < retrieval_success_prob else R_RETRIEVAL_FAILURE
        
        # Sample generator action given retrieval
        actions = [A_ANSWER, A_ABSTAIN, A_GUESS]
        probs = [action_probs_by_retrieval[(r, a)] for a in actions]
        # Normalize just in case
        total = sum(probs)
        probs = [p / total for p in probs]
        a = np.random.choice(actions, p=probs)
        
        # Sample correctness given (r, a)
        # For A_ABSTAIN, correctness is not really applicable, but we model it as 0 (incorrect) for simplicity
        # The model treats abstention as "not correct" for task success
        if a == A_ABSTAIN:
            c = C_INCORRECT  # Abstention is not a correct answer
        else:
            p_correct = correctness_probs_by_ra[(r, a, C_CORRECT)]
            p_incorrect = correctness_probs_by_ra[(r, a, C_INCORRECT)]
            total = p_correct + p_incorrect
            if total > 0:
                p_correct = p_correct / total
                p_incorrect = p_incorrect / total
            else:
                p_correct = 0.5
                p_incorrect = 0.5
            c = C_CORRECT if np.random.random() < p_correct else C_INCORRECT
        
        data.append((r, a, c))
    return data

# System A: Conservative
# - Retrieval success: 0.8
# - If retrieval success: Mostly Answer, some Abstain, little Guess
# - If retrieval failure: Mostly Abstain, little Answer/Guess
retrieval_prob_a = 0.8
action_probs_a = {
    (R_RETRIEVAL_SUCCESS, A_ANSWER): 0.8,
    (R_RETRIEVAL_SUCCESS, A_ABSTAIN): 0.1,
    (R_RETRIEVAL_SUCCESS, A_GUESS): 0.1,
    (R_RETRIEVAL_FAILURE, A_ANSWER): 0.1,
    (R_RETRIEVAL_FAILURE, A_ABSTAIN): 0.7,
    (R_RETRIEVAL_FAILURE, A_GUESS): 0.2,
}
correctness_probs_a = {
    (R_RETRIEVAL_SUCCESS, A_ANSWER, C_CORRECT): 0.9,
    (R_RETRIEVAL_SUCCESS, A_ANSWER, C_INCORRECT): 0.1,
    (R_RETRIEVAL_SUCCESS, A_ABSTAIN, C_CORRECT): 0.0,
    (R_RETRIEVAL_SUCCESS, A_ABSTAIN, C_INCORRECT): 1.0,
    (R_RETRIEVAL_SUCCESS, A_GUESS, C_CORRECT): 0.5,
    (R_RETRIEVAL_SUCCESS, A_GUESS, C_INCORRECT): 0.5,
    (R_RETRIEVAL_FAILURE, A_ANSWER, C_CORRECT): 0.2,
    (R_RETRIEVAL_FAILURE, A_ANSWER, C_INCORRECT): 0.8,
    (R_RETRIEVAL_FAILURE, A_ABSTAIN, C_CORRECT): 0.0,
    (R_RETRIEVAL_FAILURE, A_ABSTAIN, C_INCORRECT): 1.0,
    (R_RETRIEVAL_FAILURE, A_GUESS, C_CORRECT): 0.4,
    (R_RETRIEVAL_FAILURE, A_GUESS, C_INCORRECT): 0.6,
}

# System B: Aggressive
# - Retrieval success: 0.6
# - If retrieval success: Mostly Answer
# - If retrieval failure: Still tries to Answer/Guess, rarely abstains
retrieval_prob_b = 0.6
action_probs_b = {
    (R_RETRIEVAL_SUCCESS, A_ANSWER): 0.9,
    (R_RETRIEVAL_SUCCESS, A_ABSTAIN): 0.05,
    (R_RETRIEVAL_SUCCESS, A_GUESS): 0.05,
    (R_RETRIEVAL_FAILURE, A_ANSWER): 0.5,
    (R_RETRIEVAL_FAILURE, A_ABSTAIN): 0.1,
    (R_RETRIEVAL_FAILURE, A_GUESS): 0.4,
}
correctness_probs_b = {
    (R_RETRIEVAL_SUCCESS, A_ANSWER, C_CORRECT): 0.9,
    (R_RETRIEVAL_SUCCESS, A_ANSWER, C_INCORRECT): 0.1,
    (R_RETRIEVAL_SUCCESS, A_ABSTAIN, C_CORRECT): 0.0,
    (R_RETRIEVAL_SUCCESS, A_ABSTAIN, C_INCORRECT): 1.0,
    (R_RETRIEVAL_SUCCESS, A_GUESS, C_CORRECT): 0.5,
    (R_RETRIEVAL_SUCCESS, A_GUESS, C_INCORRECT): 0.5,
    (R_RETRIEVAL_FAILURE, A_ANSWER, C_CORRECT): 0.3,
    (R_RETRIEVAL_FAILURE, A_ANSWER, C_INCORRECT): 0.7,
    (R_RETRIEVAL_FAILURE, A_ABSTAIN, C_CORRECT): 0.0,
    (R_RETRIEVAL_FAILURE, A_ABSTAIN, C_INCORRECT): 1.0,
    (R_RETRIEVAL_FAILURE, A_GUESS, C_CORRECT): 0.5,
    (R_RETRIEVAL_FAILURE, A_GUESS, C_INCORRECT): 0.5,
}

n_samples = 1000
data_a = generate_simulated_data(n_samples, retrieval_prob_a, action_probs_a, correctness_probs_a)
data_b = generate_simulated_data(n_samples, retrieval_prob_b, action_probs_b, correctness_probs_b)

# Compute marginal task success (c=1) for both
success_rate_a = np.mean([1 for (r, a, c) in data_a if c == C_CORRECT])
success_rate_b = np.mean([1 for (r, a, c) in data_b if c == C_CORRECT])
print(f"System A (Conservative) Task Success Rate: {success_rate_a:.3f}")
print(f"System B (Aggressive)   Task Success Rate: {success_rate_b:.3f}")

# 2. Update Bayesian Models with Data

We initialize two `RAG_BayesianModel` instances and update them with the simulated data.
This updates the posterior distributions based on the observed frequencies.

In [ ]:
def update_model_from_data(model, data):
    """
    Update the Bayesian model's posteriors based on observed data.
    Uses MLE with Laplace smoothing.
    """
    for (r, a, c) in data:
        # Update counts for P(r)
        model.counts_r[r] += 1
        
        # Update counts for P(a|r)
        model.counts_ra[(r, a)] += 1
        
        # Update counts for P(c|r,a)
        # For abstention, we still record the outcome (c=0 typically)
        model.counts_rac[(r, a, c)] += 1
    
    # Update posteriors with smoothing
    epsilon = 1e-6
    
    # P(r)
    total_r = sum(model.counts_r.values()) + epsilon * 2
    for r in [R_RETRIEVAL_SUCCESS, R_RETRIEVAL_FAILURE]:
        model.posterior_retrieval[r] = (model.counts_r[r] + epsilon) / total_r
    
    # P(a|r)
    for r in [R_RETRIEVAL_SUCCESS, R_RETRIEVAL_FAILURE]:
        total_ar = sum(model.counts_ra[(r, a)]. for a in [A_ANSWER, A_ABSTAIN, A_GUESS]) 
        # Correction: use proper sum
        total_ar = sum(model.counts_ra[(r, a)] for a in [A_ANSWER, A_ABSTAIN, A_GUESS])
        for a in [A_ANSWER, A_ABSTAIN, A_GUESS]:
            model.posterior_action_given_retrieval[(r, a)] = (model.counts_ra[(r, a)] + epsilon) / (total_ar + 3 * epsilon)
    
    # P(c|r,a)
    for r in [R_RETRIEVAL_SUCCESS, R_RETRIEVAL_FAILURE]:
        for a in [A_ANSWER, A_ABSTAIN, A_GUESS]:
            total_rac = model.counts_rac[(r, a, C_CORRECT)] + model.counts_rac[(r, a, C_INCORRECT)]
            model.posterior_correctness_given_ra[(r, a, C_CORRECT)] = (model.counts_rac[(r, a, C_CORRECT)] + epsilon) / (total_rac + 2 * epsilon)
            model.posterior_correctness_given_ra[(r, a, C_INCORRECT)] = 1.0 - model.posterior_correctness_given_ra[(r, a, C_CORRECT)]

model_a = RAG_BayesianModel()
model_b = RAG_BayesianModel()
update_model_from_data(model_a, data_a)
update_model_from_data(model_b, data_b)

print("\nPosterior P(r=1):")
print(f"System A: {model_a.posterior_retrieval[R_RETRIEVAL_SUCCESS]:.3f}")
print(f"System B: {model_b.posterior_retrieval[R_RETRIEVAL_SUCCESS]:.3f}")

print("\nPosterior P(Abstain | r=0):")
print(f"System A: {model_a.posterior_action_given_retrieval[(R_RETRIEVAL_FAILURE, A_ABSTAIN)]:.3f}")
print(f"System B: {model_b.posterior_action_given_retrieval[(R_RETRIEVAL_FAILURE, A_ABSTAIN)]:.3f}")

# 3. Visualization: Comparing Conditional Behaviors

We create a figure with three subplots:
1. **Retrieval Success Rate**: Posterior P(r=1) for both systems.
2. **Generator Action Distribution**: P(a|r) for r=0 and r=1, shown as grouped bar charts.
3. **Answer Correctness**: P(c=1|r,a) for non-abstain actions.

This reveals how the systems behave differently despite similar end-to-end success.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Subplot 1: Retrieval Success
ax = axes[0]
systems = ['System A\n(Conservative)', 'System B\n(Aggressive)']
retrieval_succ = [model_a.posterior_retrieval[R_RETRIEVAL_SUCCESS], model_b.posterior_retrieval[R_RETRIEVAL_SUCCESS]]
retrieval_fail = [model_a.posterior_retrieval[R_RETRIEVAL_FAILURE], model_b.posterior_retrieval[R_RETRIEVAL_FAILURE]]

x = np.arange(len(systems))
width = 0.35
bars1 = ax.bar(x - width/2, retrieval_succ, width, label='Success', color='green')
bars2 = ax.bar(x + width/2, retrieval_fail, width, label='Failure', color='red')

ax.set_xlabel('System')
ax.set_ylabel('Probability')
ax.set_title('Posterior Retrieval Success Rate')
ax.set_xticks(x)
ax.set_xticklabels(systems)
ax.legend()
ax.set_ylim(0, 1)
for bar in bars1 + bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')

# Subplot 2: Generator Action Distribution (Grouped by Retrieval Outcome)
ax = axes[1]
actions = ['Answer', 'Abstain', 'Guess']
action_keys = [A_ANSWER, A_ABSTAIN, A_GUESS]

# For r=1
succ_probs_a = [model_a.posterior_action_given_retrieval[(R_RETRIEVAL_SUCCESS, a)] for a in action_keys]
fail_probs_a = [model_a.posterior_action_given_retrieval[(R_RETRIEVAL_FAILURE, a)] for a in action_keys]
succ_probs_b = [model_b.posterior_action_given_retrieval[(R_RETRIEVAL_SUCCESS, a)] for a in action_keys]
fail_probs_b = [model_b.posterior_action_given_retrieval[(R_RETRIEVAL_FAILURE, a)] for a in action_keys]

x = np.arange(len(actions))
width = 0.2
bars1 = ax.bar(x - 1.5*width, succ_probs_a, width, label='A: r=1', color='blue', alpha=0.7)
bars2 = ax.bar(x - 0.5*width, fail_probs_a, width, label='A: r=0', color='cyan', alpha=0.7)
bars3 = ax.bar(x + 0.5*width, succ_probs_b, width, label='B: r=1', color='orange', alpha=0.7)
bars4 = ax.bar(x + 1.5*width, fail_probs_b, width, label='B: r=0', color='red', alpha=0.7)

ax.set_xlabel('Generator Action')
ax.set_ylabel('Probability')
ax.set_title('Posterior Action Distribution P(a|r)')
ax.set_xticks(x)
ax.set_xticklabels(actions)
ax.legend(loc='upper right', fontsize='small')
ax.set_ylim(0, 1)
for bars in [bars1, bars2, bars3, bars4]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize='small')

# Subplot 3: Answer Correctness (for non-abstain actions)
ax = axes[2]
# Only plot for Answer and Guess actions
actions_ca = ['Answer', 'Guess']
action_keys_ca = [A_ANSWER, A_GUESS]

# For r=1
correct_ans_a = model_a.posterior_correctness_given_ra[(R_RETRIEVAL_SUCCESS, A_ANSWER, C_CORRECT)]
correct_gu_a = model_a.posterior_correctness_given_ra[(R_RETRIEVAL_SUCCESS, A_GUESS, C_CORRECT)]
correct_ans_b = model_b.posterior_correctness_given_ra[(R_RETRIEVAL_SUCCESS, A_ANSWER, C_CORRECT)]
correct_gu_b = model_b.posterior_correctness_given_ra[(R_RETRIEVAL_SUCCESS, A_GUESS, C_CORRECT)]

# For r=0
correct_ans_a0 = model_a.posterior_correctness_given_ra[(R_RETRIEVAL_FAILURE, A_ANSWER, C_CORRECT)]
correct_gu_a0 = model_a.posterior_correctness_given_ra[(R_RETRIEVAL_FAILURE, A_GUESS, C_CORRECT)]
correct_ans_b0 = model_b.posterior_correctness_given_ra[(R_RETRIEVAL_FAILURE, A_ANSWER, C_CORRECT)]
correct_gu_b0 = model_b.posterior_correctness_given_ra[(R_RETRIEVAL_FAILURE, A_GUESS, C_CORRECT)]

x = np.arange(len(actions_ca))
width = 0.2
bars1 = ax.bar(x - 1.5*width, [correct_ans_a, correct_gu_a], width, label='A: r=1', color='blue', alpha=0.7)
bars2 = ax.bar(x - 0.5*width, [correct_ans_a0, correct_gu_a0], width, label='A: r=0', color='cyan', alpha=0.7)
bars3 = ax.bar(x + 0.5*width, [correct_ans_b, correct_gu_b], width, label='B: r=1', color='orange', alpha=0.7)
bars4 = ax.bar(x + 1.5*width, [correct_ans_b0, correct_gu_b0], width, label='B: r=0', color='red', alpha=0.7)

ax.set_xlabel('Generator Action')
ax.set_ylabel('P(Correct | r, a)')
ax.set_title('Posterior Answer Correctness P(c=1|r,a)')
ax.set_xticks(x)
ax.set_xticklabels(actions_ca)
ax.legend(loc='upper right', fontsize='small')
ax.set_ylim(0, 1.2)
for bars in [bars1, bars2, bars3, bars4]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize='small')

plt.tight_layout()
plt.savefig('rag_bayesian_demo.png', dpi=150, bbox_inches='tight')
plt.show()

# 4. Key Insight: Same Success, Different Behavior

The plot above clearly shows:
- **System A** has higher retrieval success and tends to **abstain** when retrieval fails.
- **System B** has lower retrieval success but continues to **answer/guess** even when retrieval fails.
- Despite similar marginal task success, their **policy adherence** differs significantly.
- This demonstrates the value of the RAT framework: it reveals failure modes that marginal metrics miss.

In [ ]:
print("Summary of Behavioral Differences:")
print("-" * 40)
print(f"System A (Conservative):")
print(f"  - Retrieval Success: {model_a.posterior_retrieval[R_RETRIEVAL_SUCCESS]:.2f}")
print(f"  - Abstention Rate (given r=0): {model_a.posterior_action_given_retrieval[(R_RETRIEVAL_FAILURE, A_ABSTAIN)]:.2f}")
print(f"  - Correctness (Answer, r=0): {model_a.posterior_correctness_given_ra[(R_RETRIEVAL_FAILURE, A_ANSWER, C_CORRECT)]:.2f}")
print()
print(f"System B (Aggressive):")
print(f"  - Retrieval Success: {model_b.posterior_retrieval[R_RETRIEVAL_SUCCESS]:.2f}")
print(f"  - Abstention Rate (given r=0): {model_b.posterior_action_given_retrieval[(R_RETRIEVAL_FAILURE, A_ABSTAIN)]:.2f}")
print(f"  - Correctness (Answer, r=0): {model_b.posterior_correctness_given_ra[(R_RETRIEVAL_FAILURE, A_ANSWER, C_CORRECT)]:.2f}")